In [ ]:
import pandas as pd

ruta = "https://github.com/valery0906/Valery-y-Saray---bienestar-emocional/raw/refs/heads/main/EmoSPeech_phase_2_train_public.csv"

df = pd.read_csv(ruta)
df.head()

In [ ]:
df.info()

Limpieza de datos

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [ ]:
df.columns

Eliminar filas o columnas completamente vacias

In [ ]:
df.dropna(axis = 0, how = "all", inplace = True) # Fila = 0
df.dropna(axis = 1, how = "all", inplace = True) # Columna = 1

In [ ]:
df.shape #(filas, columnas)

Eliminar filas o columnas a mi eleccion

In [ ]:
cols_borrar = [c for c in ['id', 'unnamed:_0'] if c in df.columns]
df.drop(columns = cols_borrar, inplace = True)

In [ ]:
df.columns

Limpiar columnas de texto y verificar datos

In [ ]:
df.head(3)

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': pd.NA, 'None': pd.NA, '': pd.NA})

In [ ]:
df.head(3)

Convertir variables a int o float

In [ ]:
df.info()

In [ ]:
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    df[col] = pd.to_numeric(df[col], errors = "coerce")

df.info()

In [ ]:
df.head()

Quitar duplicados

In [ ]:
df.drop_duplicates(inplace = True)

In [ ]:
df.shape

Datos nulos

In [ ]:
df.isnull().sum()

Rellenar valores faltantes tipo object

In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna("No registra")

In [ ]:
df.isnull().sum()

Rellenar valores faltantes numericos, no tipo object

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

nums = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if len(nums) > 0:
    col_num = nums[0]
else:
    col_txt = df.select_dtypes(include='object').columns[0]
    df['longitud_texto'] = df[col_txt].astype(str).str.len()
    col_num = 'longitud_texto'

plt.figure(figsize=(8,5))

sns.set_style("whitegrid")  # activa cuadricula
sns.histplot(
    df[col_num],
    kde=True,
    color="#62109F",  # color solicitado
    alpha=0.7
)

plt.title(f"Distribución de {col_num}", fontsize=16)
plt.xlabel(col_num)
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x=df[col_num], color ="#62109F")
plt.title(f"Bloxplot {col_num}", fontsize=16)
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
#mean = Promedio = media
#median = mediana
media = df[col_num].mean()
mediana = df[col_num].median()

print(f"Media={media}")
print(f"Mediana={mediana}")

df[col_num] = df[col_num].fillna(mediana)

In [ ]:
df.isnull().sum()

In [ ]:
#Completar datos con la mediana
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)

In [ ]:
df.isnull().sum()

Exportar el dataset limpio

In [ ]:
import pandas as pd

df.to_excel("bienestar_emocional_limpio.xlsx", index=False)
df.to_csv("bienestar_emocional_limpio.csv", index=False)

Filtros y consultas

In [ ]:
import pandas as pd

ruta = "bienestar_emocional_limpio.csv"

df = pd.read_csv(ruta)
df.head()

In [ ]:
cols_consulta = df.columns[:3].tolist()
consulta = df[cols_consulta]

consulta.to_excel("consulta_bienestar_emocional.xlsx", index=False)

In [ ]:
ref = df[col_num].median()
df[df[col_num] < ref]

In [ ]:
cats = [c for c in df.select_dtypes(include='object').columns if df[c].nunique() < 30]
col_cat = cats[-1] if len(cats) > 0 else df.select_dtypes(include='object').columns[0]
cat_principal = df[col_cat].mode()[0]

df[df[col_cat] == cat_principal]

In [ ]:
top_cats = df[col_cat].value_counts().head(2).index.tolist()
df[df[col_cat].isin(top_cats)]

Saber cuales datos unicos tenemos en la columna

In [ ]:
df[col_cat].unique()

In [ ]:
q25 = df[col_num].quantile(0.25)
q75 = df[col_num].quantile(0.75)
df[df[col_num].between(q25, q75)] #para intervalos numericos

In [ ]:
df[cols_consulta][df[col_num].between(q25, q75)]

In [ ]:
df[col_cat].value_counts()

In [ ]:
resultado = df.groupby(col_cat).agg(
    cantidad=(col_num, 'count'),
    promedio=(col_num, 'mean')
).reset_index()

print(resultado)

In [ ]:
cols_grupo = cats[:2] if len(cats) >= 2 else [col_cat]
resultado = df[cols_grupo].drop_duplicates().reset_index(drop=True)

print(resultado)

In [ ]:
resultado = (
    df[cols_grupo]
    .drop_duplicates()
    .sort_values(cols_grupo)
    .reset_index(drop=True)
)

print(resultado)

Fin de la practica